# PyHoloscope Tutorial 4: Auto-Focus for In-Line Holography

This notebook demonstrates how to use PyHoloscope to automatically focus an inline hologram.

(If you are not familiar with Juypter notebooks you can select *Run -> Run All Cells* to see the output).

In [ ]:
from matplotlib import pyplot as plt
import sys; sys.path.append('..\src')   # Allows us to find PyHoloscope if not pip installed
import pyholoscope as pyh

An example hologram is saved as a tif file - this is what is captured by the camera. We can load this saved hologram using a convenience function in PyHoloscope, and then we display it:

In [ ]:
hologram = pyh.load_image(r"../example_data/inline_paramecium/paramecium.tif")
background = pyh.load_image(r"../example_data/inline_paramecium/background.tif")

plt.figure(dpi=200, figsize = (3,3)); plt.imshow(hologram, cmap='gray')

## Autofocus
To demodulate and recover the phase, we need to know what the spatial frequency of the modulation is. PyHoloscope can calculate this either from the image or, ideally, a background image with nothing in the field of view.

We specify the wavelength and pixel size, and also the focus scoring method. Here we use 'sum', but we can pick from a range of methods, such as 'dark_focus' and 'sobel'.

In [ ]:
wavelength = 520e-9
pixel_size = 4.8e-6 
depth_range = (.08, .16)    # Will look for focus in this range only
method = 'sum'              # Can be changed to other methods
roi = pyh.Roi(450,110,500,400)

holo = pyh.Holo(mode = pyh.INLINE, 
                wavelength = wavelength, 
                pixel_size = pixel_size, 
                background = background)

holo.set_find_focus_parameters(depth_range = depth_range, roi = roi, method = method)


We then reconstruct the hologram and display the amplitude:

In [ ]:
refocused = holo.auto_focus(hologram)

plt.figure(dpi=200, figsize = (3,3))
plt.title("Autofocused Image")
plt.imshow(pyh.amp(refocused), cmap="gray", interpolation="none")



If we know the distance from focus, we can instruct PyHoloscope to numerically refocus. Here we must also specify the wavelength and pixel size.

## Using a Propagator Look Up Table
Further customisation is possible, including only refocusing the region of interest (with a defined margin) and using a propagator look-up-table to speed up the search.


In [ ]:
margin = 50       # Refocus margin around ROI
num_depths = 100  # How many propagators in the LUT


holo2 = pyh.Holo(mode = pyh.INLINE, 
                wavelength = wavelength, 
                pixel_size = pixel_size, 
                background = background)

holo2.make_auto_focus_propagator_LUT(hologram, depth_range, num_depths, roi=roi, margin=margin)
holo2.set_find_focus_parameters(depth_range=depth_range, method=method, roi=roi, margin=margin, use_prop_lut = True)

refocused = holo2.auto_focus(hologram)

plt.figure(dpi=200, figsize = (3,3))
plt.title("Autofocused Image")
plt.imshow(pyh.amp(refocused), cmap="gray", interpolation="none")




We can compare the time just to find the focus (``auto_focus`` also refocuses the image):

In [ ]:
print("Standard:")
%timeit depth1 = holo.find_focus(hologram)
print("")
print("Partial Refocus and Propagator LUT:")
%timeit depth2 = holo2.find_focus(hologram)


The numerical outputs are similar:

In [ ]:
depth1 = holo.find_focus(hologram)
depth2 = holo2.find_focus(hologram)
print(f"Standard: {round(depth1,3)} mm, Fast: {round(depth2,3)} mm")